# Course Study Assistant

An interactive notebook for studying course material from YouTube video lectures.

## How to Use

1. **Run all code cells below** to load the tools
2. **Ask the AI** to load a YouTube video (paste the URL)
3. **Ask questions** about the content — the AI will cite timestamps and offer to embed the video
4. **Create experiments** — the AI can scaffold code cells for you to explore concepts
5. **Load references** — fastbook chapters and fastai docs are available on demand

## 1. Setup & Imports

In [ ]:
import os, re, json, subprocess, hashlib, tempfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from tqdm import tqdm
import yt_dlp
import tiktoken
from IPython.display import IFrame, display, HTML
from dialoghelper.core import *

CACHE_DIR = Path.home() / '.dialeng' / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# LLM provider abstraction — uses claudette (sync) for parallel transcript processing,
# falls back to direct Anthropic SDK if claudette is unavailable.

_llm_client = None
_llm_backend = None

def _init_llm():
    """Initialize the LLM client (lazy, called once)."""
    global _llm_client, _llm_backend
    if _llm_client is not None:
        return

    # Try claudette first (supports Anthropic API + AWS Bedrock)
    try:
        from claudette import Client, Chat
        _llm_client = Client()
        _llm_backend = 'claudette'
        print('LLM backend: claudette')
        return
    except Exception:
        pass

    # Fall back to direct Anthropic SDK
    try:
        from anthropic import Anthropic
        _llm_client = Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
        _llm_backend = 'anthropic'
        print('LLM backend: anthropic SDK')
        return
    except Exception:
        pass

    raise RuntimeError('No LLM provider available. Install claudette or set ANTHROPIC_API_KEY.')


def llm_complete(prompt: str, max_tokens: int = 8000, model: str = 'claude-sonnet-4-20250514') -> str:
    """Call the LLM synchronously. Used by transcript processing (runs in threads)."""
    _init_llm()

    if _llm_backend == 'claudette':
        response = _llm_client(model, [{'role': 'user', 'content': prompt}], maxtok=max_tokens)
        # claudette returns the message content directly
        if hasattr(response, 'content'):
            return response.content[0].text if isinstance(response.content, list) else str(response.content)
        return str(response)

    elif _llm_backend == 'anthropic':
        response = _llm_client.messages.create(
            model=model,
            max_tokens=max_tokens,
            messages=[{'role': 'user', 'content': prompt}]
        )
        return response.content[0].text


_init_llm()

## 2. Transcript Pipeline

In [ ]:
#| export

# Global state for the loaded course
_course_data = {}


def download_video_and_transcript(video_id, lang_code='en'):
    """Download video (720p) and transcript via yt_dlp."""
    video_path = f'{video_id}.mp4'
    ydl_opts = {
        'format': 'best[height<=720]',
        'quiet': True,
        'noprogress': True,
        'no_warnings': True,
        'outtmpl': video_path,
        'writesubtitles': True,
        'subtitlesformat': 'srt',
        'subtitleslangs': [lang_code],
        'writeautomaticsub': True,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(f'https://www.youtube.com/watch?v={video_id}', download=False)
        if not Path(video_path).exists():
            print('Downloading video & transcript')
            with yt_dlp.YoutubeDL(ydl_opts) as ydl2:
                ydl2.download([f'https://www.youtube.com/watch?v={video_id}'])
            print(f'Downloaded: {video_path}')
        else:
            print(f'Video already exists: {video_path}')
    return info


def parse_srt_to_text(srt_data):
    """Extract text from SRT, preserving timing."""
    lines = srt_data.strip().split('\n')
    segments = []
    current_time = None
    current_text = []
    for line in lines:
        if '-->' in line:
            current_time = line.split(' --> ')[0]
        elif line.strip() and not line.strip().isdigit():
            current_text.append(line.strip())
        elif line.strip() == '' and current_text:
            segments.append({'time': current_time, 'text': ' '.join(current_text)})
            current_text = []
    return segments


def get_seconds(time_str):
    """Convert HH:MM:SS,mmm or MM:SS,mmm to seconds."""
    parts = time_str.replace(',', '.').split(':')
    if len(parts) == 3:
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
    else:
        return int(parts[0]) * 60 + float(parts[1])


def chunk_transcript(segments, chunk_size_minutes=5, overlap_seconds=15):
    """Split transcript into time-based chunks with overlap."""
    chunks = []
    current_chunk = []
    chunk_start_time = 0
    for seg in segments:
        seconds = get_seconds(seg['time'])
        current_chunk.append(seg)
        if seconds >= chunk_start_time + (chunk_size_minutes * 60):
            chunks.append(current_chunk.copy())
            overlap_start = seconds - overlap_seconds
            current_chunk = [s for s in current_chunk if get_seconds(s['time']) >= overlap_start]
            chunk_start_time = seconds
    if current_chunk:
        chunks.append(current_chunk)
    return chunks


def count_tokens(text):
    """Count tokens using tiktoken."""
    encoding = tiktoken.get_encoding('cl100k_base')
    token_count = len(encoding.encode(text))
    print(f'Transcript Token Count: {token_count:,}')
    return token_count

In [ ]:
#| export

def process_chunk(chunk_segments, chunk_index, total_chunks):
    """Process a single transcript chunk with the LLM."""
    chunk_text = '\n'.join([f"[{seg['time']}] {seg['text']}" for seg in chunk_segments])
    start_time = chunk_segments[0]['time']
    end_time = chunk_segments[-1]['time']

    prompt = f"""You are analyzing a video transcript chunk (part {chunk_index+1} of {total_chunks}).
Time range: {start_time} to {end_time}

**Task 1: Identify chapter breaks**
If this chunk contains a major topic change that warrants a new chapter, identify it.

**Task 2: Break into sections**
Break this chunk into logical sections based on:
- Topic changes or new concepts
- Significant new information
- Roughly 90 to 120 second sections if no natural breaks

For each section provide:
- Section number
- Section title
- Start timestamp (format: MM:SS or HH:MM:SS)
- Speaker name if identifiable (or "Speaker" if not)
- Complete text as a well-formed paragraph with proper punctuation
- Chapter title (if this section starts a new chapter, otherwise null)

**Important rules:**
- Do NOT edit words - only add punctuation and capitalization
- Complete sentences only
- No overlapping sections
- Technical videos are information-dense, create granular sections

**Transcript:**
{chunk_text}

Return ONLY valid JSON array:
[
  {{
    "section": 1,
    "section_title": "Section Title",
    "timestamp": "00:00",
    "speaker": "Speaker Name",
    "chapter_title": "Chapter Title or null",
    "text": "Full paragraph text."
  }}
]"""

    response_text = llm_complete(prompt)
    return {
        'chunk_index': chunk_index,
        'response': response_text,
        'start_time': start_time,
        'end_time': end_time,
    }


def process_all_chunks(chunks, max_workers=32):
    """Process all chunks in parallel."""
    print(f'Processing {len(chunks)} chunks...')
    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(process_chunk, chunk, i, len(chunks)): i
            for i, chunk in enumerate(chunks)
        }
        for future in tqdm(as_completed(futures), total=len(chunks), desc='Processing chunks'):
            try:
                results.append(future.result())
            except Exception as e:
                print(f'Error processing chunk {futures[future]}: {e}')
    results.sort(key=lambda x: x['chunk_index'])
    print(f'Processed {len(results)} chunks successfully')
    return results


def parse_and_clean_json(response_text):
    """Extract JSON from LLM response, handling markdown code blocks."""
    if '```json' in response_text:
        response_text = response_text.split('```json')[1].split('```')[0].strip()
    elif '```' in response_text:
        response_text = response_text.split('```')[1].split('```')[0].strip()
    return json.loads(response_text)


def stitch_results(results):
    """Stitch chunk results into chapters & sections."""
    all_sections = []
    seen_texts = set()
    for result in results:
        try:
            sections = parse_and_clean_json(result['response'])
            for section in sections:
                text_key = section['text'][:100]
                if text_key not in seen_texts:
                    seen_texts.add(text_key)
                    all_sections.append(section)
        except Exception as e:
            print(f"Error parsing chunk {result['chunk_index']}: {e}")

    for i, section in enumerate(all_sections, 1):
        section['section'] = i

    chapters = []
    current_chapter = None
    for section in all_sections:
        if section.get('chapter_title'):
            if current_chapter:
                chapters.append(current_chapter)
            current_chapter = {'title': section['chapter_title'], 'sections': [section]}
        else:
            if current_chapter:
                current_chapter['sections'].append(section)
            else:
                current_chapter = {'title': 'Introduction', 'sections': [section]}
    if current_chapter:
        chapters.append(current_chapter)

    print(f'Total sections: {len(all_sections)}')
    print(f'Total chapters: {len(chapters)}')
    return chapters, all_sections

In [ ]:
#| export

def extract_screenshot(video_path, timestamp, section_num, output_dir):
    """Extract a screenshot at the given timestamp using ffmpeg."""
    try:
        seconds = get_seconds(timestamp)
        output_path = output_dir / f'screenshot_{section_num}.jpg'
        cmd = ['ffmpeg', '-ss', str(seconds), '-i', video_path,
               '-vframes', '1', '-q:v', '2', '-y', str(output_path)]
        subprocess.run(cmd, capture_output=True, check=True)
        return str(output_path)
    except Exception as e:
        print(f'Error extracting screenshot for section {section_num}: {e}')
        return None


def extract_all_screenshots(all_sections, video_path, max_workers=16):
    """Extract all screenshots in parallel."""
    output_dir = Path(f'video_screenshots/{Path(video_path).stem}')
    output_dir.mkdir(exist_ok=True, parents=True)
    screenshot_paths = {}
    print(f'Extracting {len(all_sections)} screenshots...')
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(extract_screenshot, video_path, s['timestamp'], s['section'], output_dir): s['section']
            for s in all_sections
        }
        for future in tqdm(as_completed(futures), total=len(all_sections), desc='Screenshots', leave=False):
            section_num = futures[future]
            try:
                path = future.result()
                if path:
                    screenshot_paths[section_num] = path
            except Exception as e:
                print(f'Error for section {section_num}: {e}')
    print(f'Extracted {len(screenshot_paths)}/{len(all_sections)} screenshots')
    return screenshot_paths


def format_timestamp_link(timestamp, video_id):
    """Convert timestamp to YouTube link."""
    seconds = int(get_seconds(timestamp))
    return f'https://www.youtube.com/watch?v={video_id}&t={seconds}s'


def add_transcript_msg_by_chapters(video_id, chapters, screenshots, info):
    """Add transcript as separate note messages per chapter."""
    header_md = f"# Video Transcript: {info.get('title', 'Video')}\n"
    header_md += f"**Video ID**: {video_id}\n"
    header_md += f"**Duration**: {info.get('duration_string', 'Unknown')}\n"
    header_md += f"**Total Chapters**: {len(chapters)}\n"
    add_msg(header_md, msg_type='note', placement='add_after')

    for chapter_num, chapter in enumerate(tqdm(chapters, desc='Chapters'), 1):
        chapter_lines = [f"## Chapter {chapter_num}: {chapter['title']}\n"]
        for section in chapter['sections']:
            section_num = section['section']
            yt_link = format_timestamp_link(section['timestamp'], video_id)
            chapter_lines.append(f"\n### {section['section_title']} \n")
            chapter_lines.append(
                f'<a href="{yt_link}" target="_blank">{section["timestamp"]}</a> '
                f'**{section["speaker"]}**: {section["text"]}\n'
            )
            if section_num in screenshots:
                chapter_lines.append(f'![{section["section_title"]}]({screenshots[section_num]}#ai) \n')
        chapter_md = '\n'.join(chapter_lines)
        add_msg(chapter_md, msg_type='note', placement='at_end', skipped=True, i_collapsed=True, o_collapsed=True)

    print(f'Complete! Added {len(chapters) + 1} messages to dialog')

## 3. Study Tools

In [ ]:
#| export

def load_course_video(video_url: str, lang_code: str = 'en', chunk_minutes: int = 5) -> str:
    """Download a YouTube video, extract its transcript with chapter/section structure, and add to the notebook.

    Args:
        video_url: YouTube video URL or video ID
        lang_code: Language code for transcript (default: 'en')
        chunk_minutes: Size of transcript chunks for processing (default: 5)

    Returns:
        Summary of what was loaded
    """
    global _course_data

    # Extract video ID
    if 'youtube.com' in video_url or 'youtu.be' in video_url:
        match = re.search(r'(?:v=|/)([a-zA-Z0-9_-]{11})', video_url)
        video_id = match.group(1) if match else video_url
    else:
        video_id = video_url

    print(f'Processing video: {video_id}')

    # Download
    info = download_video_and_transcript(video_id, lang_code)

    # Parse SRT
    srt_path = f'{video_id}.{lang_code}.srt'
    with open(srt_path, 'rb') as f:
        srt_data = f.read().decode('utf-8')
    segments = parse_srt_to_text(srt_data)
    full_transcript = '\n'.join([f"[{seg['time']}] {seg['text']}" for seg in segments])
    count_tokens(full_transcript)

    # Chunk and process
    chunks = chunk_transcript(segments, chunk_size_minutes=chunk_minutes)
    results = process_all_chunks(chunks)
    chapters, all_sections = stitch_results(results)

    # Screenshots
    screenshots = extract_all_screenshots(all_sections, f'{video_id}.mp4')

    # Store in global state
    _course_data = {
        'video_id': video_id,
        'info': info,
        'chapters': chapters,
        'all_sections': all_sections,
        'screenshots': screenshots,
    }

    # Add to dialog
    add_transcript_msg_by_chapters(video_id, chapters, screenshots, info)

    title = info.get('title', video_id)
    duration = info.get('duration_string', '?')
    return f'Loaded "{title}" ({duration}): {len(chapters)} chapters, {len(all_sections)} sections'

In [ ]:
#| export

def search_transcript(query: str, max_results: int = 5) -> str:
    """Search the loaded video transcript for sections matching a query.

    Args:
        query: Text to search for in the transcript
        max_results: Maximum number of results to return (default: 5)

    Returns:
        Matching transcript sections with timestamps and YouTube links
    """
    if not _course_data:
        return 'No video loaded. Use load_course_video first.'

    query_lower = query.lower()
    video_id = _course_data['video_id']
    chapters = _course_data['chapters']

    # Build a section -> chapter mapping
    section_chapter = {}
    for ch in chapters:
        for sec in ch['sections']:
            section_chapter[sec['section']] = ch['title']

    # Search in section text and titles
    matches = []
    for sec in _course_data['all_sections']:
        text = sec['text'].lower()
        title = sec['section_title'].lower()
        if query_lower in text or query_lower in title:
            matches.append(sec)

    if not matches:
        # Fuzzy: match any word from the query
        words = query_lower.split()
        for sec in _course_data['all_sections']:
            text = sec['text'].lower()
            if any(w in text for w in words if len(w) > 3):
                matches.append(sec)

    matches = matches[:max_results]

    if not matches:
        return f'No results found for "{query}" in the transcript.'

    lines = [f'Found {len(matches)} result(s) for "{query}":\n']
    for sec in matches:
        ch_title = section_chapter.get(sec['section'], '?')
        yt_link = format_timestamp_link(sec['timestamp'], video_id)
        snippet = sec['text'][:200] + ('...' if len(sec['text']) > 200 else '')
        lines.append(f'**[Chapter: {ch_title}] {sec["section_title"]}** ({sec["timestamp"]})')
        lines.append(f'{sec["speaker"]}: {snippet}')
        lines.append(f'YouTube: {yt_link}\n')
    return '\n'.join(lines)


def list_chapters() -> str:
    """List all chapters from the loaded video transcript.

    Returns:
        Numbered list of chapter titles with time ranges
    """
    if not _course_data:
        return 'No video loaded. Use load_course_video first.'

    lines = [f'Chapters for: {_course_data["info"].get("title", "Video")}\n']
    for i, ch in enumerate(_course_data['chapters'], 1):
        secs = ch['sections']
        start = secs[0]['timestamp'] if secs else '?'
        end = secs[-1]['timestamp'] if secs else '?'
        lines.append(f'{i}. **{ch["title"]}** ({start} - {end}) [{len(secs)} sections]')
    return '\n'.join(lines)


def get_chapter(chapter_num: int) -> str:
    """Get the full transcript text for a specific chapter.

    Args:
        chapter_num: Chapter number (1-indexed)

    Returns:
        Full chapter transcript with section titles and timestamps
    """
    if not _course_data:
        return 'No video loaded. Use load_course_video first.'

    chapters = _course_data['chapters']
    if chapter_num < 1 or chapter_num > len(chapters):
        return f'Invalid chapter number. Available: 1-{len(chapters)}'

    ch = chapters[chapter_num - 1]
    video_id = _course_data['video_id']
    lines = [f'## Chapter {chapter_num}: {ch["title"]}\n']
    for sec in ch['sections']:
        yt_link = format_timestamp_link(sec['timestamp'], video_id)
        lines.append(f'### {sec["section_title"]} ({sec["timestamp"]})')
        lines.append(f'{sec["speaker"]}: {sec["text"]}')
        lines.append(f'Link: {yt_link}\n')
    return '\n'.join(lines)

In [ ]:
#| export

def play_video_at(timestamp: str) -> str:
    """Embed the loaded YouTube video at a specific timestamp in the notebook.

    Args:
        timestamp: Time to start at (e.g., '12:34' or '1:02:30')

    Returns:
        Confirmation message
    """
    if not _course_data:
        return 'No video loaded. Use load_course_video first.'

    video_id = _course_data['video_id']
    seconds = int(get_seconds(timestamp))
    embed_url = f'https://www.youtube.com/embed/{video_id}?start={seconds}'
    yt_link = f'https://www.youtube.com/watch?v={video_id}&t={seconds}s'

    # Create an HTML embed to add as a note
    embed_md = (
        f'### Video at {timestamp}\n'
        f'<a href="{yt_link}" target="_blank">Open in YouTube at {timestamp}</a>\n\n'
        f'<iframe width="800" height="450" src="{embed_url}" '
        f'frameborder="0" allowfullscreen></iframe>'
    )
    add_msg(embed_md, msg_type='note', placement='add_after')
    return f'Embedded video at {timestamp} ({yt_link})'

In [ ]:
#| export

def _cached_fetch(key, fetch_fn):
    """Fetch with local file cache."""
    cache_file = CACHE_DIR / f'{hashlib.md5(key.encode()).hexdigest()}.txt'
    if cache_file.exists():
        return cache_file.read_text()
    result = fetch_fn()
    cache_file.write_text(result)
    return result


# Fastbook chapter filenames (from github.com/fastai/fastbook)
_FASTBOOK_CHAPTERS = {
    1: '01_intro.ipynb',
    2: '02_production.ipynb',
    3: '03_ethics.ipynb',
    4: '04_mnist_basics.ipynb',
    5: '05_pet_breeds.ipynb',
    6: '06_multicat.ipynb',
    7: '07_sizing_and_tta.ipynb',
    8: '08_collab.ipynb',
    9: '09_tabular.ipynb',
    10: '10_nlp.ipynb',
    11: '11_midlevel_data.ipynb',
    12: '12_nlp_dive.ipynb',
    13: '13_convolutions.ipynb',
    14: '14_resnet.ipynb',
    15: '15_arch_details.ipynb',
    16: '16_accel_sgd.ipynb',
    17: '17_foundations.ipynb',
    18: '18_CAM.ipynb',
    19: '19_learner.ipynb',
    20: '20_conclusion.ipynb',
}


def load_fastbook_chapter(chapter_num: int) -> str:
    """Load a chapter from the fastai fastbook as context for answering questions.

    Args:
        chapter_num: Chapter number (1-20)

    Returns:
        Chapter content formatted as text
    """
    if chapter_num not in _FASTBOOK_CHAPTERS:
        return f'Invalid chapter number. Available: 1-{len(_FASTBOOK_CHAPTERS)}'

    filename = _FASTBOOK_CHAPTERS[chapter_num]
    url = f'https://raw.githubusercontent.com/fastai/fastbook/master/{filename}'

    def fetch():
        import httpx
        print(f'Downloading fastbook chapter {chapter_num}: {filename}...')
        resp = httpx.get(url, follow_redirects=True, timeout=30)
        resp.raise_for_status()
        nb = json.loads(resp.text)

        # Convert notebook cells to text (similar to nbs2ctx)
        parts = [f'# Fastbook Chapter {chapter_num}: {filename}\n']
        for cell in nb.get('cells', []):
            source = ''.join(cell.get('source', []))
            if cell['cell_type'] == 'markdown':
                parts.append(source)
            elif cell['cell_type'] == 'code':
                parts.append(f'```python\n{source}\n```')
        return '\n\n'.join(parts)

    return _cached_fetch(f'fastbook_ch{chapter_num}', fetch)


# Fix contextkit/contextpack compatibility: read_url exists in contextkit
# but isn't in __all__, so contextpack's `from contextkit import *` misses it
try:
    import contextkit
    if 'read_url' not in contextkit.__all__:
        contextkit.__all__.append('read_url')
except ImportError:
    pass


def load_docs(topic: str) -> str:
    """Load documentation for a fastai ecosystem topic.

    Args:
        topic: One of 'fastcore', 'fasthtml', 'fastlite', 'docker', 'claudette'

    Returns:
        Documentation content as text
    """
    topic_lower = topic.lower().strip()

    def fetch():
        try:
            import contextpack
            accessor = getattr(contextpack, f'ctx_{topic_lower}', None)
            if accessor is None:
                return f'Topic "{topic}" not found in contextpack. Try: fastcore, fasthtml, fastlite, docker, claudette'
            # Iterate sub-topics and concatenate their content
            parts = []
            for st in accessor:
                try:
                    parts.append(st.get())
                except Exception as e:
                    parts.append(f'(Error fetching {st.url}: {e})')
            if parts:
                return '\n\n---\n\n'.join(parts)
            return str(accessor)
        except Exception as e:
            return f'Error loading docs for "{topic}": {e}'

    return _cached_fetch(f'docs_{topic_lower}', fetch)

In [ ]:
#| export

def create_experiment(title: str, starter_code: str = '', description: str = '') -> str:
    """Create a new experiment section in the notebook with a description and code cell.

    Args:
        title: Title for the experiment
        starter_code: Python code to put in the code cell (optional)
        description: Markdown description of what to experiment with (optional)

    Returns:
        Confirmation message
    """
    note_md = f'## Experiment: {title}\n'
    if description:
        note_md += f'\n{description}\n'
    add_msg(note_md, msg_type='note', placement='at_end')

    if starter_code:
        add_msg(starter_code, msg_type='code', placement='at_end')

    return f'Created experiment: "{title}"'


def search_notebook(query: str) -> str:
    """Search the current notebook for cells matching a query.

    Args:
        query: Regex pattern to search for in cell content

    Returns:
        Matching cells with index, type, and content snippet
    """
    results = find_msgs(re_pattern=query)
    if not results:
        return f'No cells found matching "{query}"'

    lines = [f'Found {len(results)} cell(s) matching "{query}":\n']
    for idx, cell_info in enumerate(results[:10]):
        # find_msgs returns list of dicts with id, content, etc.
        if isinstance(cell_info, dict):
            content = cell_info.get('content', '')
            ctype = cell_info.get('type', '?')
            cid = cell_info.get('id', '?')
        else:
            content = str(cell_info)
            ctype = '?'
            cid = '?'
        snippet = content[:150].replace('\n', ' ')
        lines.append(f'[{ctype}] {snippet}...')
    return '\n'.join(lines)

# Course Study Assistant — System Instructions

You are a course study assistant helping students learn from video lectures and associated materials.

## Your Tools

- &`load_course_video`: Load a YouTube video and extract structured transcript with chapters
- &`search_transcript`: Search the loaded transcript for specific topics
- &`play_video_at`: Embed the video at a specific timestamp in the notebook
- &`list_chapters`: List all chapters from the loaded transcript
- &`get_chapter`: Get the full transcript text for a specific chapter
- &`load_fastbook_chapter`: Load a fastbook chapter (1-20) as context
- &`load_docs`: Load fastai ecosystem docs (fastcore, fasthtml, fastai, fastlite)
- &`create_experiment`: Create new experiment cells in the notebook
- &`search_notebook`: Search the current notebook for content

## How to Respond

- When answering about video content, always cite the **chapter name**, **section title**, and **timestamp**
- Offer to embed the video at the relevant point using `play_video_at`
- When the student wants to code something, use `create_experiment` to scaffold it
- Load external references (fastbook, docs) only when needed — don't pre-load everything
- Keep answers grounded in the actual transcript and course material
- If you're not sure where something is discussed, use `search_transcript` first
- Use `list_chapters` to give an overview, then `get_chapter` for details

Start by loading a YouTube course video. For example:

"Load this video: https://www.youtube.com/watch?v=8SF_h3xF3cE"